In [0]:
from pyspark.sql import functions as F
from datetime import datetime

batch_id = datetime.now().strftime("%Y%m%d%H%M%S")

print("Data Quality Validation Started")
print(f"Batch ID: {batch_id}")

In [0]:
# ============================================================
# DATA QUALITY CHECK 1 — NULL / MANDATORY FIELD VALIDATION
# ============================================================

mandatory_columns = {
    "customers": [
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "registration_date",
        "customer_status"
    ],
    
    "customer_addresses": [
        "address_id",
        "customer_id",
        "address_type",
        "city",
        "state",
        "postal_code"
    ],
    
    "customer_contacts": [
        "contact_id",
        "customer_id",
        "contact_type",
        "contact_value"
    ],
    
    "mobile_plans": [
        "plan_id",
        "plan_name",
        "plan_type",
        "monthly_charge"
    ],
    
    "service_types": [
        "service_id",
        "service_name",
        "service_category"
    ],
    
    "plan_services": [
        "plan_service_id",
        "plan_id",
        "service_id"
    ],
    
    "subscriptions": [
        "subscription_id",
        "customer_id",
        "plan_id",
        "phone_number",
        "subscription_status",
        "activation_date"
    ],
    
    "subscription_services": [
        "subscription_service_id",
        "subscription_id",
        "service_id",
        "service_status",
        "activation_date"
    ],
    
    "call_records": [
        "call_id",
        "subscription_id",
        "call_type",
        "call_direction",
        "call_start_time",
        "call_end_time",
        "call_duration_seconds"
    ],
    
    "sms_records": [
        "sms_id",
        "subscription_id",
        "sms_type",
        "sms_direction",
        "sms_timestamp",
        "sms_count"
    ],
    
    "data_usage": [
        "usage_id",
        "subscription_id",
        "usage_date",
        "usage_start_time",
        "usage_end_time",
        "data_consumed_mb"
    ],
    
    "bills": [
        "bill_id",
        "customer_id",
        "subscription_id",
        "bill_date",
        "billing_period_start",
        "billing_period_end",
        "total_amount",
        "net_amount",
        "due_date",
        "bill_status"
    ],
    
    "bill_items": [
        "bill_item_id",
        "bill_id",
        "item_type",
        "description",
        "quantity",
        "unit_price",
        "amount"
    ],
    
    "payments": [
        "payment_id",
        "bill_id",
        "customer_id",
        "payment_date",
        "payment_amount",
        "payment_method",
        "payment_status"
    ],
    
    "complaint_categories": [
        "category_id",
        "category_name"
    ],
    
    "complaints": [
        "complaint_id",
        "customer_id",
        "category_id",
        "complaint_date",
        "complaint_status",
        "priority"
    ],
    
    "service_areas": [
        "area_id",
        "area_name",
        "city",
        "state",
        "region"
    ]
}

null_results = []

for table_name, columns in mandatory_columns.items():

    df = spark.table(f"telecom.bronze.{table_name}")

    for column_name in columns:

        null_count = df.filter(
            F.col(column_name).isNull() |
            (F.trim(F.col(column_name)) == "")
        ).count()

        null_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "null_count": null_count,
            "status": "PASS" if null_count == 0 else "FAIL"
        })

null_df = spark.createDataFrame(null_results)

display(
    null_df.orderBy(
        F.desc("null_count"),
        "table_name",
        "column_name"
    )
)

In [0]:
# ============================================================
# DATA QUALITY CHECK 2 — DUPLICATE / PRIMARY KEY VALIDATION
# ============================================================

primary_keys = {
    "customers": ["customer_id"],
    "customer_addresses": ["address_id"],
    "customer_contacts": ["contact_id"],
    "mobile_plans": ["plan_id"],
    "service_types": ["service_id"],
    "plan_services": ["plan_service_id"],
    "subscriptions": ["subscription_id"],
    "subscription_services": ["subscription_service_id"],
    "call_records": ["call_id"],
    "sms_records": ["sms_id"],
    "data_usage": ["usage_id"],
    "bills": ["bill_id"],
    "bill_items": ["bill_item_id"],
    "payments": ["payment_id"],
    "complaint_categories": ["category_id"],
    "complaints": ["complaint_id"],
    "service_areas": ["area_id"]
}

duplicate_results = []

for table_name, pk_columns in primary_keys.items():

    df = spark.table(f"telecom.bronze.{table_name}")

    total_count = df.count()

    distinct_count = (
        df.select(*pk_columns)
        .distinct()
        .count()
    )

    duplicate_count = total_count - distinct_count

    duplicate_results.append({
        "table_name": table_name,
        "total_records": total_count,
        "distinct_pk_records": distinct_count,
        "duplicate_records": duplicate_count,
        "status": "PASS" if duplicate_count == 0 else "FAIL"
    })

duplicate_df = spark.createDataFrame(duplicate_results)

display(
    duplicate_df.orderBy(
        F.desc("duplicate_records"),
        "table_name"
    )
)

In [0]:
# ============================================================
# DATA QUALITY CHECK 3 — NUMERIC / RANGE VALIDATION
# ============================================================

range_results = []

def add_range_check(table_name, rule_name, invalid_count):
    range_results.append({
        "table_name": table_name,
        "rule": rule_name,
        "invalid_records": int(invalid_count),
        "status": "PASS" if invalid_count == 0 else "FAIL"
    })


# ------------------------------------------------------------
# CALLS
# ------------------------------------------------------------

calls = spark.table("telecom.bronze.call_records")

add_range_check(
    "call_records",
    "call_duration_seconds >= 0",
    calls.filter(
        F.col("call_duration_seconds").cast("double") < 0
    ).count()
)


# ------------------------------------------------------------
# SMS
# ------------------------------------------------------------

sms = spark.table("telecom.bronze.sms_records")

add_range_check(
    "sms_records",
    "sms_count > 0",
    sms.filter(
        F.col("sms_count").cast("double") <= 0
    ).count()
)


# ------------------------------------------------------------
# DATA USAGE
# ------------------------------------------------------------

data_usage = spark.table("telecom.bronze.data_usage")

add_range_check(
    "data_usage",
    "data_consumed_mb >= 0",
    data_usage.filter(
        F.col("data_consumed_mb").cast("double") < 0
    ).count()
)


# ------------------------------------------------------------
# MOBILE PLANS
# ------------------------------------------------------------

plans = spark.table("telecom.bronze.mobile_plans")

add_range_check(
    "mobile_plans",
    "monthly_charge >= 0",
    plans.filter(
        F.col("monthly_charge").cast("double") < 0
    ).count()
)


# ------------------------------------------------------------
# BILLS
# ------------------------------------------------------------

bills = spark.table("telecom.bronze.bills")

add_range_check(
    "bills",
    "total_amount >= 0",
    bills.filter(
        F.col("total_amount").cast("double") < 0
    ).count()
)

add_range_check(
    "bills",
    "tax_amount >= 0",
    bills.filter(
        F.col("tax_amount").cast("double") < 0
    ).count()
)

add_range_check(
    "bills",
    "discount_amount >= 0",
    bills.filter(
        F.col("discount_amount").cast("double") < 0
    ).count()
)

add_range_check(
    "bills",
    "net_amount >= 0",
    bills.filter(
        F.col("net_amount").cast("double") < 0
    ).count()
)


# ------------------------------------------------------------
# BILL ITEMS
# ------------------------------------------------------------

bill_items = spark.table("telecom.bronze.bill_items")

add_range_check(
    "bill_items",
    "quantity > 0",
    bill_items.filter(
        F.col("quantity").cast("double") <= 0
    ).count()
)

add_range_check(
    "bill_items",
    "unit_price >= 0",
    bill_items.filter(
        F.col("unit_price").cast("double") < 0
    ).count()
)

add_range_check(
    "bill_items",
    "amount >= 0",
    bill_items.filter(
        F.col("amount").cast("double") < 0
    ).count()
)


# ------------------------------------------------------------
# PAYMENTS
# ------------------------------------------------------------

payments = spark.table("telecom.bronze.payments")

add_range_check(
    "payments",
    "payment_amount > 0",
    payments.filter(
        F.col("payment_amount").cast("double") <= 0
    ).count()
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

range_df = spark.createDataFrame(range_results)

display(
    range_df.orderBy(
        "table_name",
        "rule"
    )
)

In [0]:
# ============================================================
# DATA QUALITY CHECK 4 — DATE / TIMESTAMP VALIDATION
# ============================================================

date_results = []

def add_date_check(table_name, rule, invalid_count):
    date_results.append({
        "table_name": table_name,
        "rule": rule,
        "invalid_records": int(invalid_count),
        "status": "PASS" if invalid_count == 0 else "FAIL"
    })


# ------------------------------------------------------------
# CALL RECORDS
# call_end_time must be >= call_start_time
# ------------------------------------------------------------

calls = spark.table("telecom.bronze.call_records")

add_date_check(
    "call_records",
    "call_end_time >= call_start_time",
    calls.filter(
        F.to_timestamp("call_end_time") <
        F.to_timestamp("call_start_time")
    ).count()
)


# ------------------------------------------------------------
# DATA USAGE
# usage_end_time must be >= usage_start_time
# ------------------------------------------------------------

data_usage = spark.table("telecom.bronze.data_usage")

add_date_check(
    "data_usage",
    "usage_end_time >= usage_start_time",
    data_usage.filter(
        F.to_timestamp("usage_end_time") <
        F.to_timestamp("usage_start_time")
    ).count()
)


# ------------------------------------------------------------
# SUBSCRIPTIONS
# deactivation_date >= activation_date
# ------------------------------------------------------------

subscriptions = spark.table("telecom.bronze.subscriptions")

add_date_check(
    "subscriptions",
    "deactivation_date >= activation_date",
    subscriptions.filter(
        F.col("deactivation_date").isNotNull() &
        (
            F.to_date("deactivation_date") <
            F.to_date("activation_date")
        )
    ).count()
)


# ------------------------------------------------------------
# BILLS
# billing_period_end >= billing_period_start
# ------------------------------------------------------------

bills = spark.table("telecom.bronze.bills")

add_date_check(
    "bills",
    "billing_period_end >= billing_period_start",
    bills.filter(
        F.to_date("billing_period_end") <
        F.to_date("billing_period_start")
    ).count()
)


# bill date should not be before billing period start

add_date_check(
    "bills",
    "bill_date >= billing_period_start",
    bills.filter(
        F.to_date("bill_date") <
        F.to_date("billing_period_start")
    ).count()
)


# due date should not be before bill date

add_date_check(
    "bills",
    "due_date >= bill_date",
    bills.filter(
        F.to_date("due_date") <
        F.to_date("bill_date")
    ).count()
)


# ------------------------------------------------------------
# PAYMENTS
# payment_date should not be before bill_date
# ------------------------------------------------------------

payments = spark.table("telecom.bronze.payments")

payment_bill_check = (
    payments.alias("p")
    .join(
        bills.select(
            "bill_id",
            "bill_date"
        ).alias("b"),
        F.col("p.bill_id") == F.col("b.bill_id"),
        "left"
    )
)

add_date_check(
    "payments",
    "payment_date >= bill_date",
    payment_bill_check.filter(
        F.col("b.bill_date").isNotNull() &
        (
            F.to_date("p.payment_date") <
            F.to_date("b.bill_date")
        )
    ).count()
)


# ------------------------------------------------------------
# COMPLAINTS
# resolution_date >= complaint_date
# ------------------------------------------------------------

complaints = spark.table("telecom.bronze.complaints")

add_date_check(
    "complaints",
    "resolution_date >= complaint_date",
    complaints.filter(
        F.col("resolution_date").isNotNull() &
        (
            F.to_timestamp("resolution_date") <
            F.to_timestamp("complaint_date")
        )
    ).count()
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

date_df = spark.createDataFrame(date_results)

display(
    date_df.orderBy(
        "table_name",
        "rule"
    )
)

In [0]:
# ============================================================
# DATA QUALITY CHECK 5 — REFERENTIAL INTEGRITY
# ============================================================

referential_results = []

def add_fk_check(table_name, child_column, parent_table, parent_column):
    
    child_df = spark.table(f"telecom.bronze.{table_name}")
    parent_df = spark.table(f"telecom.bronze.{parent_table}")

    invalid_count = (
        child_df
        .join(
            parent_df.select(
                F.col(parent_column).alias("_parent_key")
            ),
            child_df[child_column] == F.col("_parent_key"),
            "left_anti"
        )
        .count()
    )

    referential_results.append({
        "child_table": table_name,
        "child_column": child_column,
        "parent_table": parent_table,
        "parent_column": parent_column,
        "invalid_records": invalid_count,
        "status": "PASS" if invalid_count == 0 else "FAIL"
    })


# ------------------------------------------------------------
# CUSTOMER RELATIONSHIPS
# ------------------------------------------------------------

add_fk_check(
    "customer_addresses",
    "customer_id",
    "customers",
    "customer_id"
)

add_fk_check(
    "customer_contacts",
    "customer_id",
    "customers",
    "customer_id"
)


# ------------------------------------------------------------
# PLAN / SERVICE RELATIONSHIPS
# ------------------------------------------------------------

add_fk_check(
    "plan_services",
    "plan_id",
    "mobile_plans",
    "plan_id"
)

add_fk_check(
    "plan_services",
    "service_id",
    "service_types",
    "service_id"
)


# ------------------------------------------------------------
# SUBSCRIPTION RELATIONSHIPS
# ------------------------------------------------------------

add_fk_check(
    "subscriptions",
    "customer_id",
    "customers",
    "customer_id"
)

add_fk_check(
    "subscriptions",
    "plan_id",
    "mobile_plans",
    "plan_id"
)

add_fk_check(
    "subscription_services",
    "subscription_id",
    "subscriptions",
    "subscription_id"
)

add_fk_check(
    "subscription_services",
    "service_id",
    "service_types",
    "service_id"
)


# ------------------------------------------------------------
# USAGE RELATIONSHIPS
# ------------------------------------------------------------

add_fk_check(
    "call_records",
    "subscription_id",
    "subscriptions",
    "subscription_id"
)

add_fk_check(
    "sms_records",
    "subscription_id",
    "subscriptions",
    "subscription_id"
)

add_fk_check(
    "data_usage",
    "subscription_id",
    "subscriptions",
    "subscription_id"
)


# ------------------------------------------------------------
# BILLING RELATIONSHIPS
# ------------------------------------------------------------

add_fk_check(
    "bills",
    "customer_id",
    "customers",
    "customer_id"
)

add_fk_check(
    "bills",
    "subscription_id",
    "subscriptions",
    "subscription_id"
)

add_fk_check(
    "bill_items",
    "bill_id",
    "bills",
    "bill_id"
)

add_fk_check(
    "payments",
    "bill_id",
    "bills",
    "bill_id"
)

add_fk_check(
    "payments",
    "customer_id",
    "customers",
    "customer_id"
)


# ------------------------------------------------------------
# COMPLAINT RELATIONSHIPS
# ------------------------------------------------------------

add_fk_check(
    "complaints",
    "customer_id",
    "customers",
    "customer_id"
)

add_fk_check(
    "complaints",
    "category_id",
    "complaint_categories",
    "category_id"
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

referential_df = spark.createDataFrame(referential_results)

display(
    referential_df.orderBy(
        "child_table",
        "child_column"
    )
)

In [0]:
# ============================================================
# DATA QUALITY AUDIT TABLES
# ============================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS telecom.quality.dq_results (
    batch_id STRING,
    table_name STRING,
    dq_rule STRING,
    total_records BIGINT,
    invalid_records BIGINT,
    valid_records BIGINT,
    status STRING,
    validation_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS telecom.quality.rejected_records (
    batch_id STRING,
    table_name STRING,
    record_id STRING,
    dq_rule STRING,
    rejection_reason STRING,
    rejected_timestamp TIMESTAMP
)
USING DELTA
""")

print("✅ DQ audit tables created")
print("   telecom.quality.dq_results")
print("   telecom.quality.rejected_records")

In [0]:
# ============================================================
# RECORD DQ RESULTS IN AUDIT TABLE
# ============================================================

validation_timestamp = datetime.now()

# Combine all completed DQ results
all_dq_results = []

# ------------------------------------------------------------
# 1. NULL CHECK RESULTS
# ------------------------------------------------------------

for row in null_df.collect():

    all_dq_results.append({
        "batch_id": batch_id,
        "table_name": row["table_name"],
        "dq_rule": f"NULL_CHECK: {row['column_name']}",
        "total_records": 0,
        "invalid_records": int(row["null_count"]),
        "valid_records": 0,
        "status": row["status"],
        "validation_timestamp": validation_timestamp
    })


# ------------------------------------------------------------
# 2. DUPLICATE CHECK RESULTS
# ------------------------------------------------------------

for row in duplicate_df.collect():

    all_dq_results.append({
        "batch_id": batch_id,
        "table_name": row["table_name"],
        "dq_rule": "PRIMARY_KEY_DUPLICATE_CHECK",
        "total_records": int(row["total_records"]),
        "invalid_records": int(row["duplicate_records"]),
        "valid_records": int(
            row["total_records"] - row["duplicate_records"]
        ),
        "status": row["status"],
        "validation_timestamp": validation_timestamp
    })


# ------------------------------------------------------------
# 3. RANGE CHECK RESULTS
# ------------------------------------------------------------

for row in range_df.collect():

    all_dq_results.append({
        "batch_id": batch_id,
        "table_name": row["table_name"],
        "dq_rule": f"RANGE_CHECK: {row['rule']}",
        "total_records": 0,
        "invalid_records": int(row["invalid_records"]),
        "valid_records": 0,
        "status": row["status"],
        "validation_timestamp": validation_timestamp
    })


# ------------------------------------------------------------
# 4. DATE CHECK RESULTS
# ------------------------------------------------------------

for row in date_df.collect():

    all_dq_results.append({
        "batch_id": batch_id,
        "table_name": row["table_name"],
        "dq_rule": f"DATE_CHECK: {row['rule']}",
        "total_records": 0,
        "invalid_records": int(row["invalid_records"]),
        "valid_records": 0,
        "status": row["status"],
        "validation_timestamp": validation_timestamp
    })


# ------------------------------------------------------------
# 5. REFERENTIAL INTEGRITY RESULTS
# ------------------------------------------------------------

for row in referential_df.collect():

    all_dq_results.append({
        "batch_id": batch_id,
        "table_name": row["child_table"],
        "dq_rule": (
            f"FK_CHECK: "
            f"{row['child_column']} → "
            f"{row['parent_table']}.{row['parent_column']}"
        ),
        "total_records": 0,
        "invalid_records": int(row["invalid_records"]),
        "valid_records": 0,
        "status": (
            "PASS"
            if int(row["invalid_records"]) == 0
            else "FAIL"
        ),
        "validation_timestamp": validation_timestamp
    })


# ------------------------------------------------------------
# CREATE DATAFRAME
# ------------------------------------------------------------

dq_results_df = spark.createDataFrame(all_dq_results)


# ------------------------------------------------------------
# WRITE TO DELTA AUDIT TABLE
# ------------------------------------------------------------

(
    dq_results_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("telecom.quality.dq_results")
)

print(f"✅ Recorded {dq_results_df.count():,} DQ validation results")

In [0]:
%sql
SELECT
    batch_id,
    table_name,
    dq_rule,
    invalid_records,
    status,
    validation_timestamp
FROM telecom.quality.dq_results
ORDER BY table_name, dq_rule;

In [0]:
# ============================================================
# CONTROLLED DATA QUALITY TEST RECORDS
# ============================================================

from pyspark.sql import Row

# Create intentionally invalid call records
dq_test_calls = [
    Row(
        call_id="DQ_BAD_001",
        subscription_id="SUB000001",
        call_type="LOCAL",
        call_direction="OUTGOING",
        destination_number="+12005550123",
        call_start_time="2026-08-20 10:00:00",
        call_end_time="2026-08-20 09:50:00",   # INVALID
        call_duration_seconds="-600",           # INVALID
        call_charges="0",
        created_at="2026-08-20 10:00:00"
    ),
    
    Row(
        call_id="DQ_BAD_002",
        subscription_id="INVALID_SUB",          # INVALID FK
        call_type="LOCAL",
        call_direction="OUTGOING",
        destination_number="+12005550124",
        call_start_time="2026-08-20 11:00:00",
        call_end_time="2026-08-20 11:05:00",
        call_duration_seconds="300",
        call_charges="0",
        created_at="2026-08-20 11:05:00"
    ),
    
    Row(
        call_id="DQ_BAD_003",
        subscription_id="SUB000002",
        call_type="LOCAL",
        call_direction="OUTGOING",
        destination_number="+12005550125",
        call_start_time="2026-08-20 12:00:00",
        call_end_time="2026-08-20 12:10:00",
        call_duration_seconds="-10",             # INVALID
        call_charges="0",
        created_at="2026-08-20 12:10:00"
    )
]

dq_test_calls_df = spark.createDataFrame(dq_test_calls)

print("Controlled DQ test records created:")
print(f"Records: {dq_test_calls_df.count()}")

display(dq_test_calls_df)

In [0]:
# ============================================================
# REJECT INVALID TEST RECORDS
# ============================================================

valid_subscriptions = (
    spark.table("telecom.bronze.subscriptions")
    .select("subscription_id")
    .distinct()
)

test_with_rules = (
    dq_test_calls_df
    .withColumn(
        "_duration",
        F.col("call_duration_seconds").cast("double")
    )
    .withColumn(
        "_start_time",
        F.to_timestamp("call_start_time")
    )
    .withColumn(
        "_end_time",
        F.to_timestamp("call_end_time")
    )
    .join(
        valid_subscriptions.withColumn("_valid_subscription", F.lit(True)),
        dq_test_calls_df.subscription_id ==
        valid_subscriptions.subscription_id,
        "left"
    )
)

rejected = (
    test_with_rules
    .filter(
        (F.col("_duration") < 0) |
        (F.col("_end_time") < F.col("_start_time")) |
        F.col("_valid_subscription").isNull()
    )
)

rejected_count = rejected.count()

print(f"Rejected records: {rejected_count}")

In [0]:
# ============================================================
# APPLY DQ RULES TO TEST RECORDS
# ============================================================

valid_subscriptions = (
    spark.table("telecom.bronze.subscriptions")
    .select(
        F.col("subscription_id").alias("_valid_subscription_id")
    )
    .distinct()
)

test_with_rules = (
    dq_test_calls_df
    .withColumn(
        "_duration",
        F.col("call_duration_seconds").cast("double")
    )
    .withColumn(
        "_start_time",
        F.to_timestamp("call_start_time")
    )
    .withColumn(
        "_end_time",
        F.to_timestamp("call_end_time")
    )
    .join(
        valid_subscriptions,
        F.col("subscription_id") ==
        F.col("_valid_subscription_id"),
        "left"
    )
)

rejected = (
    test_with_rules
    .filter(
        (F.col("_duration") < 0) |
        (F.col("_end_time") < F.col("_start_time")) |
        F.col("_valid_subscription_id").isNull()
    )
)

print(f"Rejected records: {rejected.count()}")

display(
    rejected.select(
        "call_id",
        "subscription_id",
        "_duration",
        "_start_time",
        "_end_time",
        "_valid_subscription_id"
    )
)

In [0]:
# ============================================================
# BUILD REJECTION REASONS
# ============================================================

rejected_with_reason = (
    rejected
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("_duration") < 0,
            F.lit("Negative call duration")
        )
        .when(
            F.col("_end_time") < F.col("_start_time"),
            F.lit("Call end time before start time")
        )
        .when(
            F.col("_valid_subscription_id").isNull(),
            F.lit("Invalid subscription reference")
        )
        .otherwise(
            F.lit("Unknown data quality failure")
        )
    )
)

display(
    rejected_with_reason.select(
        "call_id",
        "subscription_id",
        "rejection_reason"
    )
)

In [0]:
# ============================================================
# STORE REJECTED RECORDS
# ============================================================

rejected_output = (
    rejected_with_reason
    .select(
        F.lit(batch_id).alias("batch_id"),
        F.lit("call_records").alias("table_name"),
        F.col("call_id").alias("record_id"),
        F.lit("CALL_DATA_VALIDATION").alias("dq_rule"),
        F.col("rejection_reason").alias("rejection_reason"),
        F.current_timestamp().alias("rejected_timestamp")
    )
)

(
    rejected_output
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("telecom.quality.rejected_records")
)

print(f"✅ {rejected_output.count()} rejected records stored")

In [0]:
%sql
SELECT
    batch_id,
    table_name,
    record_id,
    dq_rule,
    rejection_reason,
    rejected_timestamp
FROM telecom.quality.rejected_records
ORDER BY rejected_timestamp DESC;